# Experimento oficial MultiHop-RAG

Convertido do script revisado. Execute as c?lulas em ordem. A ?ltima inicia o experimento completo e pode consumir GPU. Checkpoints permitem retomada.

## 1. Instala??o

Execute uma vez no Google Colab.

In [ ]:
%pip install -q datasets sentence-transformers pandas numpy tqdm psutil


## 2. Imports e configura??o do protocolo

In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import os
import platform
import random
import re
import statistics
import subprocess
import sys
import threading
import time
import unicodedata
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import parse_qsl, urlencode, urlparse, urlunparse

import numpy as np
import pandas as pd
import psutil
import torch
from datasets import load_dataset
from sentence_transformers import CrossEncoder, SentenceTransformer
from tqdm.auto import tqdm

# ----------------------------- Protocolo fixado -----------------------------
SEED = 42
N_QUESTIONS = None  # None = todas as elegíveis; 100 = amostra preliminar estratificada
TOP_K = 5
CANDIDATE_K = 20
MODEL_BI = "sentence-transformers/all-MiniLM-L6-v2"
MODEL_CE = "cross-encoder/ms-marco-MiniLM-L-6-v2"
CHUNK_TOKENS = 220
CHUNK_OVERLAP = 40
BATCH_SIZE = 64
CE_BATCH_SIZE = 16
DEGREE_EPS = 1.0
N_BOOTSTRAP = 5000
BOOTSTRAP_SEED = 42
CPU_SAMPLE_INTERVAL_SECONDS = 0.01
OUTDIR = Path("resultados_multihop_rag")
CACHE_DIR = OUTDIR / "cache"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
METHODS = ("bi_encoder", "degree_control", "cross_encoder")


# ------------------------------ Utilitários --------------------------------


## 3. Normaliza??o, amostragem e m?tricas

In [ ]:
def stable_hash(value: str) -> str:
    return hashlib.sha256(value.encode("utf-8")).hexdigest()


def json_dump_atomic(path: Path, obj) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with tmp.open("w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2, allow_nan=False)
        f.flush()
        os.fsync(f.fileno())
    tmp.replace(path)


def markdown_url_to_plain(value) -> str:
    if value is None:
        return ""
    value = str(value).strip()
    m = re.fullmatch(r"\[.*?\]\((https?://.*?)\)", value)
    if m:
        return m.group(1).strip()
    m = re.search(r"\]\((https?://[^)]+)\)", value)
    return m.group(1).strip() if m else value


def normalize_url(value) -> str:
    value = markdown_url_to_plain(value)
    if not value:
        return ""
    try:
        p = urlparse(value)
        if not p.netloc:
            return value.rstrip("/").lower()
        host = p.netloc.lower()
        if host.startswith("www."):
            host = host[4:]
        tracking = {"utm_source", "utm_medium", "utm_campaign", "utm_term",
                    "utm_content", "ref", "fbclid", "gclid"}
        query = [(k, v) for k, v in parse_qsl(p.query, keep_blank_values=True)
                 if k.lower() not in tracking]
        return urlunparse((p.scheme.lower() or "https", host,
                           re.sub(r"/+$", "", p.path) or "/", "",
                           urlencode(query), ""))
    except Exception:
        return value.strip().lower()


def norm_text(value) -> str:
    if value is None:
        return ""
    value = unicodedata.normalize("NFKC", str(value)).lower()
    return re.sub(r"\s+", " ", value).strip()


def stratified_sample(records, n, seed=SEED):
    if n is None or n >= len(records):
        return list(records)
    if n <= 0:
        return []
    rng = random.Random(seed)
    groups = defaultdict(list)
    for record in records:
        groups[record["question_type"]].append(record)
    raw = {key: n * len(items) / len(records) for key, items in groups.items()}
    alloc = {key: min(len(groups[key]), math.floor(value)) for key, value in raw.items()}
    left = n - sum(alloc.values())
    for key in sorted(groups, key=lambda k: (raw[k] - alloc[k], len(groups[k])), reverse=True):
        if left and alloc[key] < len(groups[key]):
            alloc[key] += 1
            left -= 1
    selected = []
    for key, items in groups.items():
        items = list(items)
        rng.shuffle(items)
        selected.extend(items[:alloc[key]])
    rng.shuffle(selected)
    return selected


def dcg_at_k(relevances, k):
    rel = np.asarray(list(relevances)[:k], dtype=float)
    if not len(rel):
        return 0.0
    return float(np.sum((2 ** rel - 1) / np.log2(np.arange(2, len(rel) + 2))))


def compute_metrics(ranked_doc_ids, gold_ids, k=TOP_K):
    ranked = list(ranked_doc_ids[:k])
    gold = set(gold_ids)
    retrieved = set(ranked)
    hits = [int(doc_id in gold) for doc_id in ranked]
    tp = len(retrieved & gold)
    precision = tp / k if k else 0.0  # fixed denominator TOP_K, as in protocol
    recall = tp / len(gold) if gold else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    rr = next((1.0 / rank for rank, hit in enumerate(hits, 1) if hit), 0.0)
    ideal_n = min(len(gold), k)
    ideal = [1] * ideal_n + [0] * (k - ideal_n)
    ndcg = dcg_at_k(hits, k) / dcg_at_k(ideal, k) if ideal_n else 0.0
    return {f"precision@{k}": precision, f"recall@{k}": recall,
            f"f1@{k}": f1, f"full_evidence@{k}": float(gold.issubset(retrieved)),
            f"mrr@{k}": rr, f"ndcg@{k}": ndcg,
            "n_gold": len(gold), f"n_relevant@{k}": tp}


# ---------------------------- Hardware telemetry ----------------------------


## 4. Medi??o de lat?ncia e hardware

In [ ]:
def synchronize_cuda():
    if torch.cuda.is_available():
        torch.cuda.synchronize()


class StageMonitor:
    """Stage wall time, process CPU-time delta, sampled process CPU and CUDA peak."""
    def __init__(self):
        self.proc = psutil.Process(os.getpid())

    def measure(self, fn):
        synchronize_cuda()
        allocated0 = torch.cuda.memory_allocated() if torch.cuda.is_available() else 0
        reserved0 = torch.cuda.memory_reserved() if torch.cuda.is_available() else 0
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
        cpu0 = self.proc.cpu_times()
        samples = []
        stop = threading.Event()

        def sample_cpu():
            while not stop.wait(CPU_SAMPLE_INTERVAL_SECONDS):
                try:
                    samples.append(self.proc.cpu_percent(interval=None))
                except (psutil.NoSuchProcess, psutil.AccessDenied):
                    break

        # Prime psutil's interval=None counter before background sampling.
        self.proc.cpu_percent(interval=None)
        sampler = threading.Thread(target=sample_cpu, daemon=True)
        sampler.start()
        wall0 = time.perf_counter()
        try:
            result = fn()
        finally:
            synchronize_cuda()
            wall = time.perf_counter() - wall0
            cpu1 = self.proc.cpu_times()
            stop.set()
            sampler.join(timeout=0.2)
        cpu_seconds = (cpu1.user + cpu1.system) - (cpu0.user + cpu0.system)
        allocated1 = torch.cuda.memory_allocated() if torch.cuda.is_available() else 0
        reserved1 = torch.cuda.memory_reserved() if torch.cuda.is_available() else 0
        if torch.cuda.is_available():
            peak_alloc = torch.cuda.max_memory_allocated()
            peak_reserved = torch.cuda.max_memory_reserved()
        else:
            peak_alloc = peak_reserved = allocated0 = reserved0 = allocated1 = reserved1 = 0
        return result, {
            "wall_seconds": wall,
            "process_cpu_seconds": cpu_seconds,
            "process_cpu_percent_mean_sampled": float(np.mean(samples)) if samples else None,
            "process_cpu_samples": len(samples),
            "cpu_logical_count": psutil.cpu_count(logical=True),
            "process_num_threads": self.proc.num_threads(),
            "gpu_allocated_before_mb": allocated0 / 2**20,
            "gpu_allocated_after_mb": allocated1 / 2**20,
            "gpu_allocated_peak_mb": peak_alloc / 2**20,
            "gpu_allocated_peak_increment_mb": max(0, peak_alloc - allocated0) / 2**20,
            "gpu_reserved_before_mb": reserved0 / 2**20,
            "gpu_reserved_after_mb": reserved1 / 2**20,
            "gpu_reserved_peak_mb": peak_reserved / 2**20,
            "gpu_reserved_peak_increment_mb": max(0, peak_reserved - reserved0) / 2**20,
        }


def latency_summary(frame):
    cols = ["mean", "median", "std", "p95", "min", "max"]
    rows = []
    for stage, group in frame.groupby("stage", dropna=False):
        x = group["wall_seconds"].dropna().to_numpy(float)
        rows.append({"stage": stage, "n": len(x), "mean": float(np.mean(x)),
                     "median": float(np.median(x)),
                     "std": float(np.std(x, ddof=1)) if len(x) > 1 else None,
                     "p95": float(np.quantile(x, .95)), "min": float(np.min(x)),
                     "max": float(np.max(x))})
    return pd.DataFrame(rows, columns=["stage", "n", *cols])


def paired_bootstrap_diff(df, method_a, method_b, metric, n_boot=N_BOOTSTRAP,
                          seed=BOOTSTRAP_SEED):
    pivot = df.pivot(index="question_id", columns="method", values=metric).dropna()
    if method_a not in pivot or method_b not in pivot:
        return None
    diffs = (pivot[method_a] - pivot[method_b]).to_numpy(dtype=float)
    if not len(diffs):
        return None
    rng = np.random.default_rng(seed)
    sample_means = np.empty(n_boot, dtype=float)
    for i in range(n_boot):
        sample_means[i] = rng.choice(diffs, size=len(diffs), replace=True).mean()
    low, high = np.quantile(sample_means, [.025, .975])
    return {"n_questions": int(len(diffs)), "mean_diff_a_minus_b": float(diffs.mean()),
            "ci95_low": float(low), "ci95_high": float(high),
            "interpretation": ("interval excludes zero; evidence of a nonzero paired difference"
                               if low > 0 or high < 0 else
                               "interval includes zero; no evidence of a nonzero difference; this does not establish equivalence")}


# -------------------------------- Experiment --------------------------------


## 5. Defini??o do experimento

A c?lula define `main()` sem iniciar a execu??o.

In [ ]:
def main():
    OUTDIR.mkdir(parents=True, exist_ok=True)
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
        torch.backends.cudnn.benchmark = False

    print(f"Device={DEVICE}; GPU={torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'}")
    print("Loading official dataset configurations...")
    ds_questions = load_dataset("yixuantt/MultiHopRAG", "MultiHopRAG")["train"]
    ds_corpus = load_dataset("yixuantt/MultiHopRAG", "corpus")["train"]
    required_q = {"query", "evidence_list", "question_type", "answer"}
    required_c = {"title", "source", "url", "body"}
    if not required_q.issubset(ds_questions.column_names):
        raise ValueError(f"Unexpected question schema: {ds_questions.column_names}")
    if not required_c.issubset(ds_corpus.column_names):
        raise ValueError(f"Unexpected corpus schema: {ds_corpus.column_names}")

    documents = []
    for idx, row in enumerate(ds_corpus):
        title, source, body = (str(row.get(k) or "").strip() for k in ("title", "source", "body"))
        if not body and not title:
            continue
        raw_url = row.get("url")
        documents.append({"doc_id": f"doc_{idx}", "corpus_index": idx, "title": title,
                          "source": source, "url": normalize_url(raw_url),
                          "url_raw": str(raw_url or ""), "body": body,
                          "text": (title + "\n\n" + body).strip()})
    if not documents:
        raise RuntimeError("No valid corpus documents.")
    doc_by_id = {d["doc_id"]: d for d in documents}
    by_url, by_title_source = defaultdict(list), defaultdict(list)
    for d in documents:
        if d["url"]:
            by_url[d["url"]].append(d["doc_id"])
        if norm_text(d["title"]):
            by_title_source[(norm_text(d["title"]), norm_text(d["source"]))].append(d["doc_id"])

    def map_evidence(ev):
        ev = ev or {}
        url = normalize_url(ev.get("url"))
        title, source = norm_text(ev.get("title")), norm_text(ev.get("source"))
        if url and url in by_url:
            ids = by_url[url]
            if len(ids) == 1:
                return ids[0], "url_exact"
            narrowed = [doc_id for doc_id in ids
                        if norm_text(doc_by_id[doc_id]["title"]) == title
                        and norm_text(doc_by_id[doc_id]["source"]) == source]
            if len(narrowed) == 1:
                return narrowed[0], "url_title_source"
            return None, "ambiguous_url"
        key = (title, source)
        if title and source and key in by_title_source:
            ids = by_title_source[key]
            return (ids[0], "title_source") if len(ids) == 1 else (None, "ambiguous_title_source")
        return None, "unmatched"

    question_records, evidence_rows, mapping_counts = [], [], Counter()
    for q_idx, row in enumerate(ds_questions):
        evidence = row.get("evidence_list") or []
        gold_ids, statuses = set(), []
        for ev_idx, ev in enumerate(evidence):
            doc_id, status = map_evidence(ev)
            statuses.append(status)
            mapping_counts[status] += 1
            if doc_id:
                gold_ids.add(doc_id)
            evidence_rows.append({"question_id": f"q_{q_idx}", "question_index": q_idx,
                                  "evidence_index": ev_idx, "mapping_status": status,
                                  "mapped_doc_id": doc_id, "raw_url": (ev or {}).get("url"),
                                  "normalized_url": normalize_url((ev or {}).get("url")),
                                  "title": (ev or {}).get("title"),
                                  "source": (ev or {}).get("source")})
        qtype = str(row.get("question_type") or "")
        question_records.append({"question_id": f"q_{q_idx}", "question_index": q_idx,
                                 "query": str(row.get("query") or "").strip(),
                                 "question_type": qtype, "answer": str(row.get("answer") or ""),
                                 "is_null_query": qtype == "null_query",
                                 "raw_evidence_count": len(evidence),
                                 "mapped_unique_doc_count": len(gold_ids),
                                 "gold_doc_ids": sorted(gold_ids), "mapping_statuses": statuses})
    mapping_df = pd.DataFrame(evidence_rows)
    mapping_df.to_csv(OUTDIR / "auditoria_evidencia_por_item.csv", index=False, encoding="utf-8-sig")
    duplicate_urls = {url: ids for url, ids in by_url.items() if len(ids) > 1}
    evidence_to_doc_counts = Counter(r["mapped_doc_id"] for r in evidence_rows if r["mapped_doc_id"])
    audit = {"created_at_utc": datetime.now(timezone.utc).isoformat(),
             "n_questions_total": len(question_records), "n_documents_valid": len(documents),
             "mapping_status_counts": dict(mapping_counts),
             "n_ambiguous_evidence": sum(v for k, v in mapping_counts.items() if k.startswith("ambiguous")),
             "n_unmatched_evidence": mapping_counts["unmatched"],
             "n_duplicate_normalized_corpus_urls": len(duplicate_urls),
             "duplicate_url_examples": dict(list(duplicate_urls.items())[:100]),
             "n_distinct_corpus_docs_targeted_by_multiple_evidence_rows": sum(n > 1 for n in evidence_to_doc_counts.values()),
             "n_non_null_questions": sum(not q["is_null_query"] for q in question_records),
             "n_null_query": sum(q["is_null_query"] for q in question_records),
             "eligibility_rule": "non-null, non-empty query, at least one evidence row, every evidence row mapped unambiguously; duplicate rows mapping to the same corpus document count once in gold_doc_ids and do not by themselves make a question ineligible",
             "n_eligible": sum((not q["is_null_query"]) and bool(q["query"])
                               and q["raw_evidence_count"] > 0
                               and all(s in {"url_exact", "url_title_source", "title_source"} for s in q["mapping_statuses"])
                               and bool(q["gold_doc_ids"]) for q in question_records)}
    json_dump_atomic(OUTDIR / "auditoria_mapeamento.json", audit)
    eligible = [q for q in question_records if (not q["is_null_query"]) and q["query"]
                and q["raw_evidence_count"] > 0 and q["gold_doc_ids"]
                and all(s in {"url_exact", "url_title_source", "title_source"} for s in q["mapping_statuses"])]
    if not eligible:
        raise RuntimeError("No eligible questions after evidence mapping.")
    selected = stratified_sample(eligible, N_QUESTIONS, SEED)
    pd.DataFrame([{"question_id": q["question_id"], "question_index": q["question_index"],
                   "question_type": q["question_type"], "raw_evidence_count": q["raw_evidence_count"],
                   "mapped_unique_doc_count": q["mapped_unique_doc_count"],
                   "gold_doc_ids": json.dumps(q["gold_doc_ids"])} for q in selected]).to_csv(
                       OUTDIR / "questoes_selecionadas.csv", index=False, encoding="utf-8-sig")

    bi_model = SentenceTransformer(MODEL_BI, device=DEVICE)
    ce_model = CrossEncoder(MODEL_CE, device=DEVICE)
    bi_module = getattr(bi_model, "_first_module", lambda: None)()
    bi_auto_model = getattr(bi_module, "auto_model", None) if bi_module else None
    bi_revision = getattr(getattr(bi_auto_model, "config", None), "_commit_hash", None)
    ce_auto_model = getattr(ce_model, "model", None)
    ce_revision = getattr(getattr(ce_auto_model, "config", None), "_commit_hash", None)
    tokenizer = bi_model.tokenizer
    chunks = []
    step = CHUNK_TOKENS - CHUNK_OVERLAP
    if step <= 0:
        raise ValueError("CHUNK_OVERLAP must be smaller than CHUNK_TOKENS")
    for d in tqdm(documents, desc="Chunking corpus"):
        token_ids = tokenizer.encode(d["text"], add_special_tokens=False)
        for j, start in enumerate(range(0, len(token_ids), step)):
            part_ids = token_ids[start:start + CHUNK_TOKENS]
            if not part_ids:
                break
            chunks.append({"chunk_id": f"{d['doc_id']}_c{j}", "doc_id": d["doc_id"],
                           "text": tokenizer.decode(part_ids, skip_special_tokens=True)})
            if start + CHUNK_TOKENS >= len(token_ids):
                break
    if not chunks:
        raise RuntimeError("No chunks produced.")
    corpus_fingerprint = stable_hash(MODEL_BI + "|" + str(bi_revision) + "|" + str(CHUNK_TOKENS) + "|" +
                                     str(CHUNK_OVERLAP) + "|" + "|".join(
                                         c["chunk_id"] + ":" + stable_hash(c["text"]) for c in chunks))
    emb_file = CACHE_DIR / f"chunk_embeddings_{corpus_fingerprint[:20]}.npy"
    meta_file = CACHE_DIR / f"chunk_metadata_{corpus_fingerprint[:20]}.json"
    expected_meta = [{"chunk_id": c["chunk_id"], "doc_id": c["doc_id"]} for c in chunks]
    if emb_file.exists() and meta_file.exists():
        with meta_file.open(encoding="utf-8") as f:
            if json.load(f) != expected_meta:
                raise RuntimeError("Embedding cache metadata does not match this corpus.")
        chunk_embeddings = np.load(emb_file, allow_pickle=False)
        if chunk_embeddings.shape[0] != len(chunks):
            raise RuntimeError("Embedding cache row count mismatch.")
    else:
        chunk_embeddings = bi_model.encode([c["text"] for c in chunks], batch_size=BATCH_SIZE,
                                           show_progress_bar=True, convert_to_numpy=True,
                                           normalize_embeddings=True).astype(np.float32)
        tmp_emb = emb_file.with_suffix(".npy.tmp")
        with tmp_emb.open("wb") as f:
            np.save(f, chunk_embeddings)
        tmp_emb.replace(emb_file)
        json_dump_atomic(meta_file, expected_meta)
    if chunk_embeddings.shape[0] != len(chunks):
        raise RuntimeError("Embedding/chunk length mismatch.")
    chunks_by_doc = defaultdict(list)
    for i, c in enumerate(chunks):
        chunks_by_doc[c["doc_id"]].append(i)
    doc_vectors, doc_vector_index = {}, {}
    for doc_id, idxs in chunks_by_doc.items():
        v = chunk_embeddings[idxs].mean(axis=0)
        norm = np.linalg.norm(v)
        if norm:
            v = v / norm
        doc_vector_index[doc_id] = len(doc_vectors)
        doc_vectors[doc_id] = v.astype(np.float32)

    def encode_query(query):
        return bi_model.encode([query], convert_to_numpy=True,
                               normalize_embeddings=True, show_progress_bar=False)[0].astype(np.float32)

    def retrieve_from_vector(qvec, top_k=CANDIDATE_K):
        scores = chunk_embeddings @ qvec
        best = {}
        for i, chunk in enumerate(chunks):
            score = float(scores[i])
            old = best.get(chunk["doc_id"])
            if old is None or score > old[0]:
                best[chunk["doc_id"]] = (score, i)
        candidates = []
        for doc_id, (score, chunk_idx) in sorted(best.items(), key=lambda x: x[1][0], reverse=True)[:top_k]:
            candidates.append({"doc_id": doc_id, "bi_score": score,
                               "best_chunk_idx": chunk_idx,
                               "best_chunk_text": chunks[chunk_idx]["text"]})
        return candidates

    def degree_rerank(candidates):
        """Use original chunk-retrieval bi_score as numerator; graph only supplies degree penalty."""
        ids = [c["doc_id"] for c in candidates]
        vecs = np.vstack([doc_vectors[x] for x in ids])
        adjacency = np.maximum(vecs @ vecs.T, 0.0)
        np.fill_diagonal(adjacency, 0.0)
        degrees = adjacency.sum(axis=1)
        ranked = []
        for i, c in enumerate(candidates):
            item = dict(c)
            item["degree"] = float(degrees[i])
            item["degree_score"] = float(c["bi_score"] / math.sqrt(DEGREE_EPS + degrees[i]))
            ranked.append(item)
        return sorted(ranked, key=lambda x: (x["degree_score"], x["bi_score"]), reverse=True)

    def cross_rerank(query, candidates):
        pairs = [(query, (doc_by_id[c["doc_id"]]["title"] + "\n" + c["best_chunk_text"]).strip())
                 for c in candidates]
        scores = ce_model.predict(pairs, batch_size=CE_BATCH_SIZE,
                                  show_progress_bar=False, convert_to_numpy=True)
        ranked = []
        for c, score in zip(candidates, scores):
            item = dict(c)
            item["ce_score"] = float(score)
            ranked.append(item)
        return sorted(ranked, key=lambda x: (x["ce_score"], x["bi_score"]), reverse=True)

    # Warmup (excluded from measured records), including only model forward passes.
    _ = encode_query("warmup")
    _ = ce_model.predict([("warmup", "warmup")], show_progress_bar=False, convert_to_numpy=True)
    if torch.cuda.is_available():
        synchronize_cuda()

    config_fingerprint = stable_hash(json.dumps({"corpus": corpus_fingerprint,
        "selected_ids": [q["question_id"] for q in selected],
        "models": [MODEL_BI, MODEL_CE], "model_revisions": [bi_revision, ce_revision],
        "top_k": TOP_K, "candidate_k": CANDIDATE_K, "degree_eps": DEGREE_EPS,
        "chunk_tokens": CHUNK_TOKENS, "chunk_overlap": CHUNK_OVERLAP}, sort_keys=True))
    run_dir = OUTDIR / ("preliminar" if N_QUESTIONS is not None else "oficial")
    run_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_path = run_dir / f"checkpoint_{config_fingerprint[:16]}.jsonl"
    checkpoint_rows = {}
    if checkpoint_path.exists():
        raw_lines = checkpoint_path.read_text(encoding="utf-8").splitlines()
        valid_lines = []
        for line_number, line in enumerate(raw_lines):
            if not line.strip():
                continue
            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                # A crash can interrupt the final append; preserve completed rows and
                # remove only a malformed trailing record before continuing.
                if line_number == len(raw_lines) - 1:
                    break
                raise RuntimeError(f"Corrupt checkpoint row {line_number + 1}; refusing unsafe resume.")
            if record.get("run_fingerprint") != config_fingerprint:
                raise RuntimeError("Checkpoint fingerprint mismatch.")
            qid = record["question_id"]
            if qid in checkpoint_rows:
                raise RuntimeError(f"Duplicate question in checkpoint: {qid}")
            checkpoint_rows[qid] = record
            valid_lines.append(json.dumps(record, ensure_ascii=False, allow_nan=False))
        if len(valid_lines) != sum(bool(line.strip()) for line in raw_lines):
            tmp_checkpoint = checkpoint_path.with_suffix(".jsonl.tmp")
            tmp_checkpoint.write_text("\n".join(valid_lines) + ("\n" if valid_lines else ""), encoding="utf-8")
            tmp_checkpoint.replace(checkpoint_path)

    monitor = StageMonitor()
    rows, stage_records, outputs = [], [], []
    metric_names = [f"precision@{TOP_K}", f"recall@{TOP_K}", f"f1@{TOP_K}",
                    f"full_evidence@{TOP_K}", f"mrr@{TOP_K}", f"ndcg@{TOP_K}"]
    for q in tqdm(selected, desc="Evaluating selected questions"):
        qid = q["question_id"]
        if qid in checkpoint_rows:
            rec = checkpoint_rows[qid]
        else:
            (qvec, enc_measure) = monitor.measure(lambda: encode_query(q["query"]))
            candidates, search_measure = monitor.measure(lambda: retrieve_from_vector(qvec, CANDIDATE_K))
            if not candidates:
                raise RuntimeError(f"No candidates for {qid}")
            degree_ranked, dc_measure = monitor.measure(lambda: degree_rerank(candidates))
            ce_ranked, ce_measure = monitor.measure(lambda: cross_rerank(q["query"], candidates))
            all_ranked = {"bi_encoder": candidates, "degree_control": degree_ranked,
                          "cross_encoder": ce_ranked}
            # Pipeline timing is separately measured as the sum of corresponding timed stages.
            measures = {
                "query_encoding": enc_measure, "bi_candidate_retrieval": search_measure,
                "degree_control_reranking": dc_measure, "cross_encoder_reranking": ce_measure,
                "pipeline_bi_encoder": {"wall_seconds": enc_measure["wall_seconds"] + search_measure["wall_seconds"]},
                "pipeline_bi_encoder_degree_control": {"wall_seconds": enc_measure["wall_seconds"] + search_measure["wall_seconds"] + dc_measure["wall_seconds"]},
                "pipeline_bi_encoder_cross_encoder": {"wall_seconds": enc_measure["wall_seconds"] + search_measure["wall_seconds"] + ce_measure["wall_seconds"]}}
            gold = set(q["gold_doc_ids"])
            candidate_recall = len({x["doc_id"] for x in candidates} & gold) / len(gold)
            rec = {"run_fingerprint": config_fingerprint, "question_id": qid,
                   "question_index": q["question_index"], "question_type": q["question_type"],
                   "query": q["query"], "gold_doc_ids": sorted(gold),
                   "candidate_doc_ids": [x["doc_id"] for x in candidates],
                   "candidate_recall@20": candidate_recall, "measures": measures, "methods": {}}
            for method, ranked in all_ranked.items():
                metric = compute_metrics([x["doc_id"] for x in ranked], gold, TOP_K)
                rec["methods"][method] = {"metrics": metric,
                    "ranking": [{"doc_id": x["doc_id"], "score": float(x.get(
                        "bi_score" if method == "bi_encoder" else
                        "degree_score" if method == "degree_control" else "ce_score"))}
                                for x in ranked[:TOP_K]]}
            # Append and fsync per completed question: interruption resumes without duplicate IDs.
            with checkpoint_path.open("a", encoding="utf-8") as f:
                f.write(json.dumps(rec, ensure_ascii=False, allow_nan=False) + "\n")
                f.flush()
                os.fsync(f.fileno())
            checkpoint_rows[qid] = rec
        outputs.append(rec)
        for stage, measure in rec["measures"].items():
            stage_records.append({"question_id": qid, "question_index": q["question_index"],
                                  "question_type": q["question_type"], "stage": stage, **measure})
        for method in METHODS:
            rows.append({"question_id": qid, "question_index": q["question_index"],
                         "question_type": q["question_type"], "method": method,
                         "candidate_recall@20": rec["candidate_recall@20"],
                         **rec["methods"][method]["metrics"]})

    result_df = pd.DataFrame(rows)
    stage_df = pd.DataFrame(stage_records)
    if result_df["question_id"].nunique() != len(selected) or len(result_df) != len(selected) * 3:
        raise RuntimeError("Result association/integrity check failed.")
    prefix = "preliminar" if N_QUESTIONS is not None else "oficial"
    result_df.to_csv(run_dir / f"resultados_individuais_{prefix}.csv", index=False, encoding="utf-8-sig")
    json_dump_atomic(run_dir / f"resultados_brutos_{prefix}.json", outputs)
    stage_df.to_csv(run_dir / f"telemetria_etapas_{prefix}.csv", index=False, encoding="utf-8-sig")

    summary_cols = metric_names  # shared Candidate Recall is reported once by the retrieval stage below
    per_type = result_df.groupby(["question_type", "method"])[summary_cols].agg(["mean", "std"])
    overall = result_df.groupby("method")[summary_cols].agg(["mean", "std"])
    per_type.to_csv(run_dir / f"metricas_por_tipo_{prefix}.csv")
    overall.to_csv(run_dir / f"metricas_gerais_{prefix}.csv")
    candidate_summary = result_df[["question_id", "question_type", "candidate_recall@20"]].drop_duplicates()
    candidate_summary.to_csv(run_dir / f"candidate_recall_compartilhado_{prefix}.csv", index=False, encoding="utf-8-sig")
    latency = latency_summary(stage_df)
    latency.to_csv(run_dir / f"resumo_latencia_{prefix}.csv", index=False, encoding="utf-8-sig")

    boot = {}
    for metric in metric_names:
        for a, b in (("degree_control", "bi_encoder"), ("degree_control", "cross_encoder"),
                     ("cross_encoder", "bi_encoder")):
            boot[f"{metric}:{a}-{b}"] = paired_bootstrap_diff(result_df, a, b, metric)
    json_dump_atomic(run_dir / f"bootstrap_pareado_{prefix}.json", boot)

    # Null-query diagnostics remain outside the main quality analysis.
    null_rows = []
    for q in tqdm([x for x in question_records if x["is_null_query"] and x["query"]], desc="Null-query diagnostics"):
        qvec = encode_query(q["query"])
        cands = retrieve_from_vector(qvec, CANDIDATE_K)
        null_rows.append({"question_id": q["question_id"], "question_type": q["question_type"],
                          "n_candidates": len(cands), "max_bi_similarity": max((c["bi_score"] for c in cands), default=None)})
    pd.DataFrame(null_rows).to_csv(run_dir / f"diagnostico_null_query_{prefix}.csv", index=False, encoding="utf-8-sig")

    model_revisions = {"bi_encoder": bi_revision, "cross_encoder": ce_revision}
    try:
        gpu_info = {"name": torch.cuda.get_device_name(0), "capability": torch.cuda.get_device_capability(0),
                    "total_memory_mb": torch.cuda.get_device_properties(0).total_memory / 2**20} if torch.cuda.is_available() else None
    except Exception:
        gpu_info = None
    try:
        pip_freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True,
                                    text=True, timeout=30).stdout.splitlines()
    except Exception:
        pip_freeze = None
    metadata = {"timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "dataset": "yixuantt/MultiHopRAG", "questions_config": "MultiHopRAG", "corpus_config": "corpus",
        "model_bi": MODEL_BI, "model_ce": MODEL_CE, "model_revisions_if_available": model_revisions,
        "seed": SEED, "bootstrap_seed": BOOTSTRAP_SEED, "bootstrap_resamples": N_BOOTSTRAP,
        "top_k": TOP_K, "candidate_k": CANDIDATE_K, "chunk_tokens": CHUNK_TOKENS,
        "chunk_overlap": CHUNK_OVERLAP, "degree_epsilon": DEGREE_EPS,
        "degree_control_formula": "original Bi-Encoder best-chunk score / sqrt(DEGREE_EPS + positive-cosine weighted degree); graph only computes penalty",
        "candidate_unit": "article; score=max(query cosine, chunks belonging to article)",
        "cross_encoder_input": "query paired with title + best chunk selected by Bi-Encoder",
        "n_questions_dataset": len(ds_questions), "n_documents": len(documents), "n_chunks": len(chunks),
        "n_eligible": len(eligible), "n_selected": len(selected),
        "question_type_counts_selected": dict(Counter(q["question_type"] for q in selected)),
        "evidence_counts_selected": {q["question_id"]: {"raw": q["raw_evidence_count"],
            "mapped_unique_docs": q["mapped_unique_doc_count"]} for q in selected},
        "run_fingerprint": config_fingerprint, "device": DEVICE, "gpu": gpu_info,
        "cpu_logical_count": psutil.cpu_count(logical=True), "cpu_physical_count": psutil.cpu_count(logical=False),
        "ram_total_gb": psutil.virtual_memory().total / 2**30,
        "process_threads_at_metadata": psutil.Process().num_threads(),
        "python": platform.python_version(), "platform": platform.platform(), "torch": torch.__version__,
        "cuda_runtime": torch.version.cuda, "cuda_available": torch.cuda.is_available(),
        "libraries_pip_freeze": pip_freeze,
        "latency_definition": {"query_encoding": "query vector encoding only",
            "bi_candidate_retrieval": "chunk dot products, best chunk per document and top-20 article selection; excludes query encoding",
            "reranking": "reranker only after shared query encoding and candidate retrieval",
            "pipelines": "sum of measured stage wall times; Bi = encode+retrieve; DC/CE = encode+retrieve+their reranker"},
        "cpu_measurement_limits": "Per-process CPU time is stage delta. Mean process CPU percent is sampled every 10ms; very short stages may have no sample and are null. Sampling adds minor overhead; whole-machine contention is not attributed to a method.",
        "gpu_memory_limits": "Allocated/reserved totals include resident model/cache baseline; peak increments are relative to pre-stage baseline. CUDA allocator reservation can persist between stages; this is not total device/process memory.",
        "warmup_excluded": True, "cpu_sample_interval_seconds": CPU_SAMPLE_INTERVAL_SECONDS,
        "selection_rule": audit["eligibility_rule"], "files": sorted(str(p) for p in run_dir.iterdir() if p.is_file())}
    json_dump_atomic(run_dir / f"metadados_{prefix}.json", metadata)
    print("\nCompleted. Output directory:", run_dir.resolve())
    print("Mapping audit:", OUTDIR / "auditoria_mapeamento.json")
    print("Latency summary:\n", latency.to_string(index=False))
    print("Bootstrap results:", json.dumps(boot, ensure_ascii=False, indent=2))


## 6. Executar

Esta c?lula inicia o processamento. Deixe-a sem executar para apenas preparar o notebook.

In [ ]:
main()
